# ⚽ Project 05: Player Market Valuation & Tactical Playstyle Role Clustering
### Sports Analytics, Latent Role Discovery via PCA & Regularized Linear Models

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟢 Beginner  
**Domain:** Sports Analytics & Scouting  

---
### Notebook Outline:
1. **Environment Setup**
2. **Data Ingestion & Scouting Profile Inspection**
3. **Exploratory Data Analysis: Metrics Correlation & Variance Inflation Factors (VIF)**
4. **Principal Component Analysis (PCA) for Latent Playstyle Discovery**
5. **Multi-Model Regression: OLS vs. Ridge vs. Lasso**
6. **Cross-Validation & Hyperparameter Alpha Search**
7. **Moneyball Talent Arbitrage: Finding Undervalued Gems**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Sports analytics workspace configured.")

In [ ]:
# Data Ingestion
df = pd.read_csv("data/sports_player_scouting.csv")
print(f"Scouted Athletes: {len(df)}")
display(df.head(4))

In [ ]:
# PCA for Latent Tactical Playstyle Discovery
stats_cols = ['expected_goals_xg', 'expected_assists_xa', 'pass_completion_pct', 
              'progressive_carries_p90', 'tackles_p90', 'sprint_speed_kmh']

X_stats = df[stats_cols]
scaler = StandardScaler()
X_stats_scaled = scaler.fit_transform(X_stats)

pca = PCA(n_components=2)
coords_pca = pca.fit_transform(X_stats_scaled)
df['PC1_Attacking_Pace'] = coords_pca[:, 0]
df['PC2_Defensive_Control'] = coords_pca[:, 1]

print(f"PCA Variance Explained: {pca.explained_variance_ratio_.sum():.2%}")

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, x='PC1_Attacking_Pace', y='PC2_Defensive_Control',
    hue='nominal_position', palette='Set1', s=50, alpha=0.8
)
plt.title("Latent Tactical Roles Discovered via PCA", fontweight='bold')
plt.xlabel("PC1 (Attacking Threat & Ball Carrying)")
plt.ylabel("PC2 (Defensive Workrate & Passing)")
plt.show()

In [ ]:
# Regression Modeling: Predicting Market Value
features = ['age', 'minutes_played', 'expected_goals_xg', 'expected_assists_xa',
            'pass_completion_pct', 'progressive_carries_p90', 'tackles_p90', 'sprint_speed_kmh']
X = df[features]
y = df['market_value_eur_mil']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler_reg = StandardScaler()
X_train_sc = scaler_reg.fit_transform(X_train)
X_test_sc = scaler_reg.transform(X_test)

models = {
    "OLS Linear": LinearRegression(),
    "Ridge (L2)": Ridge(alpha=1.0),
    "Lasso (L1)": Lasso(alpha=0.1)
}

results = []
for name, m in models.items():
    m.fit(X_train_sc, y_train)
    preds = m.predict(X_test_sc)
    results.append({
        "Model": name,
        "R2 Score": round(r2_score(y_test, preds), 4),
        "RMSE (Mil €)": round(np.sqrt(mean_squared_error(y_test, preds)), 3),
        "MAE (Mil €)": round(mean_absolute_error(y_test, preds), 3)
    })

display(pd.DataFrame(results))

In [ ]:
# Scouting Arbitrage: Identifying Undervalued Players
champion_lasso = Lasso(alpha=0.1).fit(scaler_reg.transform(X), y)
df['predicted_market_value'] = champion_lasso.predict(scaler_reg.transform(X))
df['market_value_delta'] = df['predicted_market_value'] - df['market_value_eur_mil']

undervalued = df.sort_values('market_value_delta', ascending=False).head(5)
print("=== Top 5 Most Undervalued Transfer Targets (Moneyball Alpha) ===")
display(undervalued[['player_id', 'age', 'nominal_position', 'market_value_eur_mil', 
                     'predicted_market_value', 'market_value_delta']].round(2))